# Thành viên 4
Các bảng phụ trách: `employees.csv`, `shippers.csv`, `shipments.csv`, `payments.csv`

Chạy lần lượt từ **Ô 0** đến ô cuối. Mỗi ô tạo đúng một file CSV, đọc từ dữ liệu gốc `student_data`.

## Ô 0: Chuẩn bị

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

DATA = Path("/content/student_data")   # dữ liệu gốc (bronze)
OUT = Path("/content/silver")          # kết quả (silver)
OUT.mkdir(exist_ok=True)

if not DATA.exists():                  # chưa có dữ liệu -> chọn file student_data.zip
    from google.colab import files
    for name in files.upload():
        zipfile.ZipFile(name).extractall("/content")

def strip(df, cols):
    """Bỏ khoảng trắng thừa ở các cột chữ."""
    for c in cols:
        df[c] = df[c].astype("string").str.strip()
    return df

def save(df, name, pk):
    """Kiểm tra khóa chính (duy nhất, không rỗng) rồi ghi CSV."""
    pk = [pk] if isinstance(pk, str) else pk
    assert df[pk].notna().all().all() and not df.duplicated(pk).any(), f"{name}: khóa chính lỗi"
    df.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}.csv: {len(df):,} dòng, khóa chính {pk} hợp lệ")
    return df.head()

Saving student_data.zip to student_data.zip


## Ô 1: `employees.csv`

In [2]:
# Tách nhân viên ra khỏi đơn hàng: sales_employee_id -> tên, hôn nhân, học vấn, kinh nghiệm
cols = ["sales_employee_id", "sales_employee_name", "marital_status", "education_level", "years_experience"]
employees = strip(pd.read_csv(DATA / "orders_enriched.csv", usecols=cols),
                  ["sales_employee_id", "sales_employee_name", "marital_status", "education_level"])
employees["sales_employee_id"] = employees["sales_employee_id"].str.upper()
assert (employees.groupby("sales_employee_id").nunique() <= 1).all().all(), "1 mã nhân viên có nhiều bộ thông tin"

save(employees.drop_duplicates("sales_employee_id").sort_values("sales_employee_id").reset_index(drop=True),
     "employees", "sales_employee_id")

employees.csv: 200 dòng, khóa chính ['sales_employee_id'] hợp lệ


,sales_employee_id,sales_employee_name,marital_status,education_level,years_experience
0,EMP0001,Trần Anh Phúc,Đã kết hôn,Đại học,9
1,EMP0002,Phạm Văn Mai,Đã kết hôn,Đại học,20
2,EMP0003,Vũ Thị Trang,Độc thân,Sau đại học,10
3,EMP0004,Đỗ Thanh Lan,Đã kết hôn,Cao đẳng,17
4,EMP0005,Trần Gia Uyên,Đã kết hôn,Đại học,2


## Ô 2: `shippers.csv`

In [3]:
# Tách shipper ra khỏi chuyến giao hàng. Sửa: bỏ region (city -> region), city đổi tên city_name làm khóa ngoại tới city
cols = ["shipper_id", "shipper_name", "shipper_phone", "shipper_gender", "shipper_age",
        "shipper_marital_status", "shipper_education", "shipper_company", "shipper_vehicle",
        "shipper_experience_years", "shipper_rating", "delivery_success_rate",
        "average_delivery_time", "working_shift", "join_date", "city", "region", "district"]
raw = pd.read_csv(DATA / "shipments_realistic.csv", usecols=cols)
strip(raw, ["shipper_id", "shipper_name", "shipper_gender", "shipper_marital_status", "shipper_education",
            "shipper_company", "shipper_vehicle", "working_shift", "city", "region", "district"])
raw["shipper_id"] = raw["shipper_id"].str.upper()
raw["join_date"] = pd.to_datetime(raw["join_date"])
assert (raw.groupby("shipper_id").nunique() <= 1).all().all(), "1 shipper có nhiều bộ thông tin"
shippers = raw.drop_duplicates("shipper_id")

# Kiểm tra city -> region khớp geography trước khi bỏ region (đồng thời đảm bảo city có trong bảng city)
geo = pd.read_csv(DATA / "geography.csv", usecols=["city", "region"]).drop_duplicates()
chk = shippers.merge(geo, on=["city", "region"], how="left", indicator=True)
assert (chk["_merge"] == "both").all(), "city/region của shipper không khớp geography"

shippers = shippers.drop(columns="region").rename(columns={"city": "city_name"})
save(shippers.sort_values("shipper_id").reset_index(drop=True), "shippers", "shipper_id")

shippers.csv: 80 dòng, khóa chính ['shipper_id'] hợp lệ


,shipper_id,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,average_delivery_time,working_shift,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city_name,district
0,SHP00001,Viettel Post,Truck,7,5.0,99.0,61,Evening,2026-03-17,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Phan Rang-Thap Cham,District #25
1,SHP00002,J&T Express,Van,2,4.9,98.4,72,Afternoon,2025-01-29,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,Phan Thiet,District #29
2,SHP00003,GHN,Motorbike,10,4.8,95.1,53,Evening,2019-11-13,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,District #34
3,SHP00004,Viettel Post,Truck,8,5.0,96.3,53,Evening,2025-12-22,Trần Đức Vy,971617475,Female,42,Married,College,Kon Tum,District #27
4,SHP00005,BEST Express,Truck,10,4.6,95.7,62,Morning,2019-12-19,Trần Minh Cường,979196342,Male,31,Married,College,Da Nang,District #23


## Ô 3: `shipments.csv`

In [4]:
cols = ["order_id", "shipper_id", "ship_date", "delivery_date", "shipping_fee"]
shipments = strip(pd.read_csv(DATA / "shipments_realistic.csv", usecols=cols), ["shipper_id"])
shipments["shipper_id"] = shipments["shipper_id"].str.upper()
for c in ["ship_date", "delivery_date"]:
    shipments[c] = pd.to_datetime(shipments[c])
assert (shipments["delivery_date"] >= shipments["ship_date"]).all(), "có ngày giao trước ngày gửi"
assert (shipments["shipping_fee"] >= 0).all(), "có phí ship âm"

save(shipments.sort_values("order_id").reset_index(drop=True), "shipments", "order_id")

shipments.csv: 566,067 dòng, khóa chính ['order_id'] hợp lệ


,shipper_id,order_id,ship_date,delivery_date,shipping_fee
0,SHP00001,1,2012-07-07,2012-07-11,1.37
1,SHP00002,2,2012-07-06,2012-07-10,2.60
2,SHP00003,3,2012-07-04,2012-07-07,2.38
3,SHP00004,4,2012-07-05,2012-07-11,2.49
4,SHP00005,6,2012-07-09,2012-07-16,25.79


## Ô 4: `payments.csv`

In [5]:
payments = strip(pd.read_csv(DATA / "payments.csv"), ["payment_method"])
payments["payment_method"] = payments["payment_method"].str.lower()
assert payments["payment_method"].isin(["credit_card", "paypal", "cod", "apple_pay", "bank_transfer"]).all()
assert (payments["payment_value"] >= 0).all() and (payments["installments"] >= 1).all()

save(payments.sort_values("order_id").reset_index(drop=True), "payments", "order_id")

payments.csv: 646,945 dòng, khóa chính ['order_id'] hợp lệ


,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1
